<a href="https://colab.research.google.com/github/ChenS1313/thelook-ecommerce-analytics-pipeline/blob/ChenS1313-patch-4/thelooker_ecommerce_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##1 Data Loading & Initial Exploration


### Loading the relevant tables from **Google BigQuery**

In [ ]:
# Import relevant libraries
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Connect Google account for BigQuery access
auth.authenticate_user()

In [ ]:
# Set up the BigQuery connection and dataset name
client = bigquery.Client(project="project-69dc4deb-819d-43d1-af9")
dataset = "dbt_csinai"

# Create dataframe for each table
df_users = client.list_rows(f"{dataset}.dim_users").to_dataframe()
df_orders = client.list_rows(f"{dataset}.dim_orders").to_dataframe()
df_products = client.list_rows(f"{dataset}.dim_products").to_dataframe()
df_order_items = client.list_rows(f"{dataset}.fct_order_items").to_dataframe()


### Checking the data structure to make sure everything loaded correctly

In [ ]:
# Verify table sizes and row counts
print(f"Users table:{df_users.shape}")
print(f"Orders table:{df_orders.shape}")
print(f"Products table:{df_products.shape}")
print(f"Order_items table:{df_order_items.shape}")

In [69]:
# Check column names and data types using.info()
datasets = {
    "Users": df_users,
    "Orders": df_orders,
    "Products": df_products,
    "Order Items": df_order_items
}

for name,df in datasets.items():
   print(f"======== {name} Table Info ========")
   df.info()
   print("\n")

======== Users Table Info ========
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 20 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   user_id                 100000 non-null  object        
 1   first_name              100000 non-null  object        
 2   last_name               100000 non-null  object        
 3   age                     100000 non-null  Int64         
 4   gender                  100000 non-null  object        
 5   email                   100000 non-null  object        
 6   address                 100000 non-null  object        
 7   city                    100000 non-null  object        
 8   state                   100000 non-null  object        
 9   country                 100000 non-null  object        
 10  postal_code             100000 non-null  object        
 11  latitude                100000 non-null  float64       
 

In [ ]:
# Inspect the data samples
df_users.head()
df_products.head()
df_order_items.head()
df_orders.head()

## 2 Data Formatting

Due to default Pandas display settings when loading data from **Google BigQuery**, we apply light formatting to improve readability without changing the actual data:

- **Dates:** Standardize date formatting.

- **Numeric Columns:** Round prices and costs to 2 decimal places.
This includes casting numeric columns back to `float64` if BigQuery's `NUMERIC` type forced them into Python `object` types during import.

- **Missing Data:** Keep empty cells as Pandas `NaT` / `NaN` / `<NA>` (which match BigQuery `NULL`s).



In [ ]:
# Format all date columns across all tables
dfs_list = [df_users, df_products, df_order_items, df_orders]
for df in dfs_list:
    for col in df.columns:
        # Check if the column is of datetime type
        if df[col].dtype == 'datetime64[us, UTC]':
                df[col] = pd.to_datetime(df[col]).dt.tz_localize(None).astype('datetime64[ns]')

In [ ]:
# Convert BigQuery NUMERIC columns from object back to float

# df_users
df_users['lifetime_value'] = pd.to_numeric(df_users['lifetime_value'], errors = 'coerce')
df_users['total_profit'] = pd.to_numeric(df_users['total_profit'], errors = 'coerce')

# df_products
df_products['sale_price'] = pd.to_numeric(df_products['sale_price'], errors = 'coerce')
df_products['cost'] = pd.to_numeric(df_products['cost'], errors = 'coerce')
df_products['total_sales'] = pd.to_numeric(df_products['total_sales'], errors = 'coerce')
df_products['total_profit'] = pd.to_numeric(df_products['total_profit'], errors = 'coerce')


# df_order_items
df_order_items['sale_price'] = pd.to_numeric(df_order_items['sale_price'], errors = 'coerce')
df_order_items['cost'] = pd.to_numeric(df_order_items['cost'], errors = 'coerce')
df_order_items['profit'] = pd.to_numeric(df_order_items['profit'], errors = 'coerce')

# df_orders
df_orders['order_sales'] = pd.to_numeric(df_orders['order_sales'], errors = 'coerce')
df_orders['order_cost'] = pd.to_numeric(df_orders['order_cost'], errors = 'coerce')
df_orders['order_profit'] = pd.to_numeric(df_orders['order_profit'], errors = 'coerce')


In [70]:
# Check that all data types were successfully updated across all datasets using.info()

for name,df in datasets.items():
   print(f"======== {name} Table Info ========")
   df.info()
   print("\n")

======== Users Table Info ========
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 20 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   user_id                 100000 non-null  object        
 1   first_name              100000 non-null  object        
 2   last_name               100000 non-null  object        
 3   age                     100000 non-null  Int64         
 4   gender                  100000 non-null  object        
 5   email                   100000 non-null  object        
 6   address                 100000 non-null  object        
 7   city                    100000 non-null  object        
 8   state                   100000 non-null  object        
 9   country                 100000 non-null  object        
 10  postal_code             100000 non-null  object        
 11  latitude                100000 non-null  float64       
 

## 3 Deep EDA & Visualizations